# Two Pointers

The first of the problem-solving **patterns** — reusable templates that combine the structures and algorithms from the earlier tiers. The two-pointer pattern walks *two* indices through a sequence instead of one, turning many brute-force **O(n²)** scans into a single **O(n)** pass with **O(1)** extra space.

> **Pattern notebooks read a little differently** from the structure/algorithm ones: **signal** (when to reach for it) → **template** (the reusable skeleton) → **worked examples** (runnable) → **when-to-use & Python toolkit**.

**Contents**

1. **The signal**
2. **Opposite ends** — converging pointers
3. **Same direction** — read / write pointers
4. **Fast & slow** — Floyd's pointers
5. **When to use & toolkit**
6. **Complexity cheat-sheet**

## 1. The signal — when to reach for two pointers

Reach for two pointers when you see:

- a **sorted** array and you want a **pair / triplet** meeting a condition (sum, difference) → *converging* pointers from both ends;
- an **in-place** transformation that compacts or partitions a sequence (remove, move, dedupe) → a *read* pointer and a *write* pointer moving the **same** direction;
- a **cycle** or a **midpoint** in a sequence or linked list → a *slow* and a *fast* pointer.

The payoff is almost always **O(n²) → O(n)** and **extra space → O(1)**, by exploiting structure: sortedness, or the fact that one pointer never needs to look back.

## 2. Opposite ends — converging pointers

Start one pointer at each end and move them **toward each other**. On a **sorted** array this is the classic two-sum: if the current pair sums too low, the only way up is to advance the left pointer; too high, retreat the right. Each step eliminates one candidate, so it's one O(n) pass instead of the O(n²) double loop.

In [1]:
def two_sum_sorted(a, target):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        s = a[lo] + a[hi]
        if s == target:
            return (lo, hi)
        elif s < target:
            lo += 1                  # need a bigger sum -> move the left end up
        else:
            hi -= 1                  # need a smaller sum -> move the right end down
    return None

print("two_sum_sorted([1,2,4,7,11,15], 15):", two_sum_sorted([1, 2, 4, 7, 11, 15], 15))

# the same converging shape verifies a palindrome:
def is_palindrome(s):
    lo, hi = 0, len(s) - 1
    while lo < hi:
        if s[lo] != s[hi]:
            return False
        lo += 1; hi -= 1
    return True

print("is_palindrome('racecar'):", is_palindrome("racecar"), "| ('hello'):", is_palindrome("hello"))


two_sum_sorted([1,2,4,7,11,15], 15): (2, 4)
is_palindrome('racecar'): True | ('hello'): False


This converging shape is also the engine behind "container with most water," **3-sum** (fix one element, two-pointer the rest), and "trapping rain water" — any time a sorted/bounded range lets you decide which end to move.

## 3. Same direction — read / write pointers

Here both pointers move the **same way**: a **read** pointer scans every element while a **write** pointer marks where the next *kept* element goes. It compacts a sequence **in place** in one pass, O(1) extra space — no second list. Dedupe a sorted array, or push every zero to the end:

In [2]:
def remove_duplicates(a):
    """Sorted array -> keep one of each, in place; returns the new length."""
    if not a:
        return 0
    w = 1                            # write index: a[:w] holds the unique prefix
    for r in range(1, len(a)):       # read index scans the rest
        if a[r] != a[w - 1]:
            a[w] = a[r]
            w += 1
    return w

arr = [1, 1, 2, 2, 2, 3]
k = remove_duplicates(arr)
print("remove_duplicates -> length", k, "| unique prefix:", arr[:k])

def move_zeroes(a):
    """Move all zeroes to the end, keeping the order of the rest, in place."""
    w = 0
    for r in range(len(a)):
        if a[r] != 0:
            a[w], a[r] = a[r], a[w]  # swap the next non-zero into the write slot
            w += 1
    return a

print("move_zeroes([0,1,0,3,12]):", move_zeroes([0, 1, 0, 3, 12]))


remove_duplicates -> length 3 | unique prefix: [1, 2, 3]
move_zeroes([0,1,0,3,12]): [1, 3, 12, 0, 0]


The read/write split is the same idea as the **partition** step in quicksort (the sorting notebook): keep a boundary index where "everything before me satisfies the predicate," and swap qualifying elements across it.

## 4. Fast & slow — Floyd's pointers

Advance one pointer **one step** and another **two steps**. If the sequence loops, the fast pointer laps the slow one and they meet (Floyd's *tortoise & hare*); if it terminates, the fast pointer reaches the end first. We used this for linked-list **cycle detection** and middle-finding in the linked-lists notebook — but it generalizes to *any* "next" function, no linked list required.

A **happy number** repeatedly replaces n with the sum of the squares of its digits; it's "happy" if that reaches 1, and otherwise loops forever. Detect the loop with fast & slow:

In [3]:
def is_happy(n):
    def step(x):
        return sum(int(d) ** 2 for d in str(x))   # the 'next' in the sequence
    slow = fast = n
    while True:
        slow = step(slow)            # one step
        fast = step(step(fast))      # two steps
        if fast == 1:
            return True              # reached the fixed point -> the sequence terminates
        if slow == fast:
            return False             # slow and fast met -> a cycle

print("is_happy(19):", is_happy(19), "| is_happy(2):", is_happy(2))


is_happy(19): True | is_happy(2): False


For an actual linked list, the same template finds the **middle** (when fast reaches the end, slow sits at the midpoint) and detects or locates a **cycle** — see the linked-lists notebook §3, where `reverse` and `has_cycle` use exactly this idea.

## 5. When to use & toolkit

| You see... | Use | Turns |
|---|---|---|
| Sorted array, find a pair/triplet by sum | converging pointers | O(n²) → O(n) |
| In-place compact / partition / dedupe | read/write pointers | O(n) time, O(1) space |
| Cycle or midpoint in a sequence / list | fast & slow | O(n) time, O(1) space |
| Merge two sorted sequences | one pointer per sequence | O(n + m) |

**Python toolkit:** `sorted(...)` is the usual precondition for the converging form; pair it with `bisect` (searching notebook) when one side should jump by binary search instead of stepping. For the read/write form, a final slice assignment like `a[w:] = []` truncates the compacted tail in place.

**Prerequisite to spot:** two pointers usually needs **sorted** input (converging) or an **order-preserving** goal (read/write). If the data isn't sorted and order doesn't matter, a one-pass `set`/`dict` (O(n)) is often simpler than sorting first.

## 6. Complexity cheat-sheet

| Variant | Time | Space | Precondition |
|---|:---:|:---:|---|
| Converging (both ends) | O(n) | O(1) | sorted input |
| Read / write (same direction) | O(n) | O(1) | — |
| Fast & slow | O(n) | O(1) | a "next" function |

<sub>All three replace an O(n²) brute force with a single linear pass and constant extra space — the recurring promise of the two-pointer pattern.</sub>